# Titanic Exploratory Data Analysis
**Project 01 · Data Science & AI · Beginner-friendly walkthrough**

This notebook follows the project guide and uses Seaborn's Titanic passenger dataset. It inspects and cleans the data, checks duplicates and data types, compares survival by sex, class, and age band, and writes three findings from computed results.

> This is a reproducible reference analysis, not the user's original Colab notebook or its exact results. Run all cells to generate outputs.

## 1. Questions to answer
1. Did survival differ by sex?
2. Did passenger class relate to survival?
3. How did survival vary across age bands?

These are descriptive questions; the patterns do not show that a factor caused survival.

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="deep")
print("Libraries imported successfully.")

## 2. Load the dataset
Seaborn downloads the small public Titanic CSV the first time it is used. An internet connection is required.

In [ ]:
df = sns.load_dataset("titanic")
print(f"Dataset shape: {df.shape[0]} rows × {df.shape[1]} columns")
display(df.head())

## 3. Inspect the data
Run the standard DataFrame checks: dimensions, sample rows, columns and types, and summary statistics.

In [ ]:
print("Shape:", df.shape)
print("\nColumn names:", df.columns.tolist())
df.info()

In [ ]:
display(df.describe(include="all").T)

### Column guide
- `survived`: 0 = did not survive, 1 = survived; `pclass`: passenger class (1 = first, 2 = second, 3 = third)
- `sex`, `age`: passenger sex and age in years
- `sibsp`, `parch`: siblings/spouses and parents/children aboard; `fare`: ticket fare
- `embarked`, `class`, `who`, `adult_male`, `deck`, `embark_town`, `alive`, `alone`: travel, port, and derived descriptors

Use `describe(include="all")` to see both numeric and categorical summaries.

## 4. Find missing values
Count missing entries and calculate each column's missing percentage before choosing how to clean.

In [ ]:
missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(1),
}).sort_values("missing_count", ascending=False)
display(missing_summary)

In [ ]:
missing_to_plot = missing_summary.loc[missing_summary.missing_count > 0, "missing_count"].sort_values()
plt.figure(figsize=(9, 4.5))
missing_to_plot.plot(kind="barh", color="#3b82a0")
plt.title("Missing values by column (before cleaning)")
plt.xlabel("Number of missing values")
plt.ylabel("Column")
plt.tight_layout()
plt.show()

## 5. Check duplicates and clean
The guide recommends filling age by class and sex, dropping mostly empty deck, and filling embarked with its mode. We report duplicate rows but do not automatically delete them: identical rows are not automatically errors.

In [ ]:
print(f"Exact duplicate rows before cleaning: {int(df.duplicated().sum())}")
print(f"Deck missing: {df.deck.isna().sum()} of {len(df)} ({df.deck.isna().mean():.1%})")
print(f"Age missing: {df.age.isna().sum()}")
print(f"Embarked missing: {df.embarked.isna().sum()}")

In [ ]:
clean_df = df.copy()
group_age_median = clean_df.groupby(["pclass", "sex"])["age"].transform("median")
clean_df["age"] = clean_df["age"].fillna(group_age_median)
clean_df["age"] = clean_df["age"].fillna(clean_df["age"].median())
embarked_mode = clean_df["embarked"].mode(dropna=True)
if not embarked_mode.empty:
    clean_df["embarked"] = clean_df["embarked"].fillna(embarked_mode.iloc[0])
clean_df = clean_df.drop(columns=["deck"])
print("Cleaned shape:", clean_df.shape)
print("Missing values remaining:", int(clean_df.isna().sum().sum()))
print("Exact duplicate rows after cleaning (reported, not removed):", int(clean_df.duplicated().sum()))

## 6. Verify data types
Confirm numeric analysis columns are numeric and convert category-like fields to pandas categories.

In [ ]:
for column in ["survived", "pclass", "age"]:
    assert pd.api.types.is_numeric_dtype(clean_df[column]), f"{column} should be numeric"
for column in ["sex", "embarked", "who", "alive"]:
    clean_df[column] = clean_df[column].astype("category")
print(clean_df.dtypes)

## 7. Compare survival by sex and class
The mean of the 0/1 `survived` column is the survival rate. Include passenger counts for context.

In [ ]:
survival_by_sex = (clean_df.groupby("sex", observed=True)["survived"]
                  .agg(survival_rate="mean", passengers="count")
                  .sort_values("survival_rate", ascending=False))
survival_by_sex["survival_percent"] = (survival_by_sex.survival_rate * 100).round(1)
display(survival_by_sex)
survival_by_class = (clean_df.groupby("pclass", observed=True)["survived"]
                     .agg(survival_rate="mean", passengers="count").sort_index())
survival_by_class["survival_percent"] = (survival_by_class.survival_rate * 100).round(1)
display(survival_by_class)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.barplot(data=clean_df, x="sex", y="survived", errorbar=None, ax=axes[0], color="#3b82a0")
axes[0].set(title="Survival rate by sex", xlabel="Sex", ylabel="Survival rate", ylim=(0, 1))
sns.barplot(data=clean_df, x="pclass", y="survived", errorbar=None, ax=axes[1], color="#e09f3e")
axes[1].set(title="Survival rate by passenger class", xlabel="Passenger class", ylabel="Survival rate", ylim=(0, 1))
plt.tight_layout()
plt.show()

## 8. Compare survival by age band
Create the guide's age bands with `pd.cut`, then group and aggregate.

In [ ]:
age_bins = [0, 12, 18, 40, 60, 100]
age_labels = ["Child (0-12)", "Teen (12-18)", "Adult (18-40)", "Middle-aged (40-60)", "Senior (60-100)"]
clean_df["age_band"] = pd.cut(clean_df.age, bins=age_bins, labels=age_labels, include_lowest=True)
survival_by_age = (clean_df.groupby("age_band", observed=True)["survived"]
                   .agg(survival_rate="mean", passengers="count"))
survival_by_age["survival_percent"] = (survival_by_age.survival_rate * 100).round(1)
display(survival_by_age)

In [ ]:
plt.figure(figsize=(9, 4.5))
sns.barplot(data=clean_df, x="age_band", y="survived", errorbar=None, color="#6b8e5a")
plt.title("Survival rate by age band")
plt.xlabel("Age band")
plt.ylabel("Survival rate")
plt.ylim(0, 1)
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

## 9. Three data-driven findings
The statements below are generated from the cleaned data in this run. Each gives a percentage and group size; they update if the dataset changes.

In [ ]:
def percent(rate):
    return f"{rate:.1%}"

sex_order = survival_by_sex.sort_values("survival_rate", ascending=False)
class_order = survival_by_class.sort_values("survival_rate", ascending=False)
age_order = survival_by_age.dropna(subset=["survival_rate"]).sort_values("survival_rate", ascending=False)
top, bottom = sex_order.iloc[0], sex_order.iloc[-1]
print(f"1. Sex: {sex_order.index[0]} passengers survived at {percent(top.survival_rate)} (n={int(top.passengers)}), compared with {percent(bottom.survival_rate)} for {sex_order.index[-1]} passengers (n={int(bottom.passengers)}).")
top, bottom = class_order.iloc[0], class_order.iloc[-1]
print(f"2. Class: class {class_order.index[0]} had a {percent(top.survival_rate)} survival rate (n={int(top.passengers)}), compared with {percent(bottom.survival_rate)} for class {class_order.index[-1]} (n={int(bottom.passengers)}).")
top, bottom = age_order.iloc[0], age_order.iloc[-1]
print(f"3. Age band: {age_order.index[0]} had the highest survival rate at {percent(top.survival_rate)} (n={int(top.passengers)}); {age_order.index[-1]} had the lowest at {percent(bottom.survival_rate)} (n={int(bottom.passengers)}).")
print("\nThese are descriptive patterns in this dataset; they do not establish that sex, class, or age caused the outcomes.")

## 10. Conclusion
This analysis inspected the Titanic dataset, measured missingness, applied documented cleaning decisions, reported duplicates, checked types, and compared survival by sex, class, and age band. Use the three numbered results above as this run's written summary. Before submission, restart the runtime and choose **Run all** to confirm every cell executes in order. You can export the notebook as HTML or PDF from Jupyter or Colab.